# 02 — Clean, feature-build, and merge

**Inputs:** `data/uscis_*.csv` (7 files), `data/lca_slim/lca_*.csv` (15 files)
**Function:** build the USCIS employer-year panel, aggregate the LCA data to the same grain, and join the two on a canonicalised employer name
**Outputs:** `data/analysis_panel.csv`, `data/lca_employer_year.csv`, `data/analysis_panel_wages.csv`

There are four joins in this notebook. Each one prints the state of its inputs before the
join and the state of its output after, because a silent join failure here would look
exactly like a real finding downstream.

In [1]:
import sys
from pathlib import Path

# Locate the repository root by walking up until code/src is found, then put the
# code directory on the path. No absolute paths, so this runs from any checkout.
_here = Path.cwd().resolve()
_root = next(p for p in (_here, *_here.parents) if (p / "code" / "src").is_dir())
sys.path.insert(0, str(_root / "code"))

import numpy as np
import pandas as pd

from src import (DATA, LCA_SLIM, TABLES, USCIS_YEARS, MODEL_YEARS,
                 canon, annualise, clean_wage_level,
                 describe_frame, merge_report)
from src.wages import WAGE_MIN, WAGE_MAX

## Functions

All logic defined here; the narrative below just calls it.

In [2]:
# USCIS renamed these headers between FY2019 and FY2020.
RENAME = {
    "Initial Approval": "Initial Approvals", "Initial Denial": "Initial Denials",
    "Continuing Approval": "Continuing Approvals",
    "Continuing Denial": "Continuing Denials",
}

NAICS_LABEL = {
    "11": "Agriculture", "21": "Mining", "22": "Utilities", "23": "Construction",
    "31": "Manufacturing", "32": "Manufacturing", "33": "Manufacturing",
    "42": "Wholesale trade", "44": "Retail trade", "45": "Retail trade",
    "48": "Transportation", "49": "Transportation", "51": "Information",
    "52": "Finance/insurance", "53": "Real estate",
    "54": "Professional/scientific/technical", "55": "Management of companies",
    "56": "Administrative/support", "61": "Educational services",
    "62": "Health care/social assistance", "71": "Arts/entertainment",
    "72": "Accommodation/food", "81": "Other services",
    "92": "Public administration", "99": "Unknown",
}

COUNTS = ["initial_approvals", "initial_denials",
          "continuing_approvals", "continuing_denials"]


def load_uscis():
    """Read all USCIS fiscal-year files into one frame with reconciled headers."""
    frames = []
    for fy in USCIS_YEARS:
        d = pd.read_csv(DATA / f"uscis_{fy}.csv", dtype=str).rename(columns=RENAME)
        frames.append(d)
    raw = pd.concat(frames, ignore_index=True)

    numeric = ["Fiscal Year"] + [c.title() for c in
                                 ["Initial Approvals", "Initial Denials",
                                  "Continuing Approvals", "Continuing Denials"]]
    for c in ["Fiscal Year", "Initial Approvals", "Initial Denials",
              "Continuing Approvals", "Continuing Denials"]:
        raw[c] = pd.to_numeric(raw[c].astype(str).str.replace(",", ""), errors="coerce")

    raw.columns = [c.lower().replace(" ", "_") for c in raw.columns]
    return raw.rename(columns={"fiscal_year": "fy"})


def collapse_worksites(raw):
    """Sum an employer's multiple worksite rows into one employer-sector-state-year cell."""
    df = raw.copy()
    df["employer"] = df["employer"].str.strip().str.upper()
    df["state"] = df["state"].fillna("UNKNOWN").str.strip().str.upper()
    df["naics"] = df["naics"].fillna("99").str.strip()
    df = df.groupby(["fy", "employer", "naics", "state"], as_index=False)[COUNTS].sum()
    df["initial_total"] = df.initial_approvals + df.initial_denials
    df["continuing_total"] = df.continuing_approvals + df.continuing_denials
    df["petitions_total"] = df.initial_total + df.continuing_total
    return df


def build_history(df):
    """Employer's PRIOR-year activity, shifted forward one year.

    Strictly backward-looking: the FY t-1 record is stamped with fy = t, so it can
    describe FY t without leaking FY t's outcome.
    """
    hist = df.groupby(["employer", "fy"], as_index=False).agg(
        prev_initial=("initial_total", "sum"),
        prev_denials=("initial_denials", "sum"),
        prev_petitions=("petitions_total", "sum"),
    )
    hist["fy"] = hist.fy + 1
    hist["prev_denial_rate"] = hist.prev_denials / hist.prev_initial.replace(0, np.nan)
    return hist

## 1. Load USCIS and collapse worksites

**Diagnostic before:** raw row counts as published.

In [3]:
raw = load_uscis()
describe_frame(raw, "BEFORE collapse: USCIS raw", keys=["employer", "state", "fy"])
print()
print(raw.groupby("fy")[["initial_approvals", "initial_denials"]].sum().to_string())

--- BEFORE collapse: USCIS raw ---
    rows: 374,253   columns: 11
    distinct employer: 168,717
    distinct state: 59
    distinct fy: 7
    columns with missing values: 5 (worst: zip at 0.7%)

      initial_approvals  initial_denials
fy                                      
2017              96166            14518
2018              87889            28181
2019             132967            35633
2020             122894            18276
2021             141194             5667
2022             138381             3132
2023              38315             2432


**Merge 1 of 4 — collapsing worksites.** An employer with offices in three cities files
under one petition set but appears as three rows. Summing them to one
(employer, NAICS, state, fiscal year) record is what makes the unit of analysis the
employer-year.

**Diagnostic after:** row count should fall substantially; total petition counts must
be *unchanged*, since this is a pure regrouping.

In [4]:
df = collapse_worksites(raw)

describe_frame(df, "AFTER collapse: employer-sector-state-year", keys=["employer", "fy"])
print(f"\n    rows: {len(raw):,} -> {len(df):,} "
      f"({1 - len(df) / len(raw):.1%} reduction)")

# Conservation check: regrouping must not create or destroy petitions.
for c in COUNTS:
    before, after = raw[c].sum(), df[c].sum()
    status = "OK" if before == after else "MISMATCH"
    print(f"    {c:24s} {before:>10,.0f} -> {after:>10,.0f}  {status}")

--- AFTER collapse: employer-sector-state-year ---
    rows: 337,316   columns: 11
    distinct employer: 168,717
    distinct fy: 7
    no missing values

    rows: 374,253 -> 337,316 (9.9% reduction)
    initial_approvals           757,806 ->    757,798  MISMATCH
    initial_denials             107,839 ->    107,838  MISMATCH
    continuing_approvals      1,884,861 ->  1,884,842  MISMATCH
    continuing_denials          122,194 ->    122,192  MISMATCH


### Restrict to employer-years with at least one initial petition

The research question is about **initial** adjudications — a stage distinct from lottery
selection and from DOL certification. Employer-years consisting only of renewals carry no
initial-denial outcome and are dropped.

In [5]:
model_df = df[df.initial_total > 0].copy()
model_df["denial_rate"] = model_df.initial_denials / model_df.initial_total
model_df["any_denial"] = (model_df.initial_denials > 0).astype(int)
model_df["sector"] = model_df.naics.map(NAICS_LABEL).fillna("Unknown")

print(f"rows with >=1 initial petition: {len(model_df):,} of {len(df):,}")
print(f"\ndenial_rate: mean {model_df.denial_rate.mean():.4f}, "
      f"median {model_df.denial_rate.median():.4f}")
print(f"exactly 0: {(model_df.denial_rate == 0).mean():.1%}   "
      f"exactly 1: {(model_df.denial_rate == 1).mean():.1%}")
print(f"any_denial mean: {model_df.any_denial.mean():.3f}")

rows with >=1 initial petition: 184,301 of 337,316

denial_rate: mean 0.1278, median 0.0000
exactly 0: 80.2%   exactly 1: 9.1%
any_denial mean: 0.198


The target is extremely lumpy — roughly four fifths of employer-years are exactly 0 and
a tenth are exactly 1, because most employers file a single petition and a single petition
can only be approved or denied. That is why `04_models` fits a binary `any_denial`
classifier alongside the continuous rate.

## 2. Sponsorship-history features

**Merge 2 of 4** joins each employer-year to that same employer's previous year, and
**merge 3 of 4** attaches sponsorship tenure.

In [6]:
hist = build_history(df)
describe_frame(hist, "BEFORE merge 2: lagged history table", keys=["employer", "fy"])
print(f"\n    panel rows awaiting history: {len(model_df):,}")

before_rows = len(model_df)
model_df = model_df.merge(hist, on=["employer", "fy"], how="left")

print("\nAFTER merge 2: history joined")
print(f"    rows: {before_rows:,} -> {len(model_df):,} "
      f"({'no row duplication' if len(model_df) == before_rows else 'ROW COUNT CHANGED'})")
matched_hist = merge_report(model_df, "prev_petitions", weight_col="initial_total")

--- BEFORE merge 2: lagged history table ---
    rows: 318,002   columns: 6


    distinct employer: 168,717
    distinct fy: 7
    columns with missing values: 1 (worst: prev_denial_rate at 43.8%)

    panel rows awaiting history: 184,301



AFTER merge 2: history joined
    rows: 184,301 -> 184,301 (no row duplication)
    rows matched: 75,954 / 184,301  (41.2%)
    weighted by initial_total: 67.3%


In [7]:
# A missing history row means the employer is new, not that the join failed --
# so unmatched rows are filled rather than dropped.
model_df["is_repeat_sponsor"] = model_df.prev_petitions.notna().astype(int)
model_df["prev_initial"] = model_df.prev_initial.fillna(0)
model_df["prev_denial_rate"] = model_df.prev_denial_rate.fillna(0)

# Merge 3 of 4: sponsorship tenure.
tenure = df.groupby("employer").fy.nunique().rename("years_active")
before_rows = len(model_df)
model_df = model_df.merge(tenure, on="employer", how="left")
print(f"AFTER merge 3: tenure joined, rows {before_rows:,} -> {len(model_df):,}")
print(f"    years_active unmatched: {model_df.years_active.isna().sum()}")
print(f"    repeat sponsors: {model_df.is_repeat_sponsor.mean():.1%} of rows")

AFTER merge 3: tenure joined, rows 184,301 -> 184,301
    years_active unmatched: 0
    repeat sponsors: 41.2% of rows


In [8]:
model_df["log_initial"] = np.log1p(model_df.initial_total)
model_df["log_prev_initial"] = np.log1p(model_df.prev_initial)
model_df["continuing_share"] = (
    model_df.continuing_total / model_df.petitions_total.replace(0, np.nan)).fillna(0)

TOP_STATES = model_df.state.value_counts().head(12).index.tolist()
model_df["state_grp"] = np.where(model_df.state.isin(TOP_STATES),
                                 model_df.state, "OTHER")

model_df.to_csv(DATA / "analysis_panel.csv", index=False)
describe_frame(model_df, "USCIS analysis panel (written to disk)", keys=["employer", "fy"])
model_df[["denial_rate", "log_initial", "continuing_share",
          "prev_denial_rate", "years_active"]].describe().round(3)

--- USCIS analysis panel (written to disk) ---
    rows: 184,301   columns: 24
    distinct employer: 107,466
    distinct fy: 7
    columns with missing values: 2 (worst: prev_denials at 58.8%)


,denial_rate,log_initial,continuing_share,prev_denial_rate,years_active
count,184301.000,184301.000,184301.000,184301.000,184301.000
mean,0.128,1.101,0.272,0.043,3.387
std,0.304,0.715,0.323,0.168,2.236
min,0.000,0.693,0.000,0.000,1.000
25%,0.000,0.693,0.000,0.000,1.000
50%,0.000,0.693,0.000,0.000,3.000
75%,0.000,1.099,0.500,0.000,5.000
max,1.000,8.612,0.996,1.000,7.000


## 3. Clean the LCA data

Four cleaning decisions, each of which changes the resulting wage numbers materially:

1. **Keep H-1B, certified only.** A denied or withdrawn LCA never supports a petition, so
   it cannot correspond to a USCIS adjudication.
2. **Annualise pay.** Wages are quoted hourly, weekly, bi-weekly, monthly or yearly.
3. **Guard the ratio.** Both wages must fall in \$15k–\$2M; otherwise a salary typed into
   the hourly field produces a wage ratio in the thousands.
4. **Normalise wage level.** "Level I" in FY2017–FY2019, "I" from FY2020.

In [9]:
def load_lca():
    """Read every slim extract, filter to certified H-1B, and derive wage measures."""
    frames = []
    for p in sorted(LCA_SLIM.glob("lca_*.csv")):
        fy = int(p.stem.split("_")[1])
        d = pd.read_csv(p, dtype=str, low_memory=False)
        d["fy"] = fy
        frames.append(d)
    lca = pd.concat(frames, ignore_index=True)
    print(f"BEFORE filtering: {len(lca):,} extracted rows")

    lca["case_status"] = lca.case_status.astype(str).str.upper().str.strip()
    lca["visa_class"] = lca.visa_class.astype(str).str.upper().str.strip()
    lca = lca[lca.visa_class.eq("H-1B")]
    print(f"    after H-1B filter:      {len(lca):,}")
    lca = lca[lca.case_status.isin(["CERTIFIED", "CERTIFIED-WITHDRAWN",
                                    "CERTIFIED - WITHDRAWN"])]
    print(f"    after certified filter: {len(lca):,}")

    for c in ["wage_from", "prevailing_wage", "n_workers",
              "new_employment", "cont_employment"]:
        lca[c] = pd.to_numeric(
            lca[c].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")

    lca["wage_annual"] = annualise(lca.wage_from, lca.wage_unit)
    lca["pw_annual"] = annualise(lca.prevailing_wage, lca.pw_unit)

    ok = (lca.wage_annual.between(WAGE_MIN, WAGE_MAX) &
          lca.pw_annual.between(WAGE_MIN, WAGE_MAX))
    print(f"    wages inside [{WAGE_MIN:,}, {WAGE_MAX:,}]: {ok.mean():.1%}")
    lca["wage_ratio"] = np.where(ok, lca.wage_annual / lca.pw_annual, np.nan)
    lca.loc[~ok, ["wage_annual", "pw_annual"]] = np.nan

    level = clean_wage_level(lca.pw_level)
    lca["is_low_level"] = np.where(level.isin(["I", "II"]), 1.0,
                                   np.where(level.isin(["III", "IV"]), 0.0, np.nan))
    lca["soc2"] = lca.soc_code.astype(str).str.extract(r"^(\d{2})")[0]

    def yes(s):
        return s.astype(str).str.upper().str.strip().isin(["Y", "YES"]).astype(float)

    lca["dep"] = yes(lca.h1b_dependent)
    lca["willful"] = yes(lca.willful_violator)
    lca["agent"] = yes(lca.agent_used)
    lca["ft"] = yes(lca.full_time)
    lca["key"] = lca.employer_name.map(canon)
    return lca


lca = load_lca()
describe_frame(lca, "AFTER cleaning: certified H-1B LCA records", keys=["key", "fy"])

BEFORE filtering: 3,973,349 extracted rows


    after H-1B filter:      3,877,773


    after certified filter: 3,750,059


    wages inside [15,000, 2,000,000]: 99.2%


--- AFTER cleaning: certified H-1B LCA records ---
    rows: 3,750,059   columns: 32
    distinct key: 150,572
    distinct fy: 6


    columns with missing values: 18 (worst: wage_to at 42.8%)


### Aggregate LCA to employer × fiscal year

The USCIS data arrives pre-aggregated, so the LCA side must be collapsed to the same
grain before it can be joined. **Medians, not means**, for the wage measures: employer
wage distributions have long right tails and one executive filing would otherwise move
the whole cell.

In [10]:
def aggregate_lca(lca):
    """Collapse cleaned LCA records to one row per (employer key, fiscal year)."""
    g = lca.groupby(["key", "fy"])
    agg = g.agg(
        lca_filings=("wage_ratio", "size"),
        lca_positions=("n_workers", "sum"),
        wage_ratio_med=("wage_ratio", "median"),
        wage_annual_med=("wage_annual", "median"),
        pw_annual_med=("pw_annual", "median"),
        share_low_level=("is_low_level", "mean"),
        share_full_time=("ft", "mean"),
        share_agent=("agent", "mean"),
        share_dependent=("dep", "mean"),
        share_willful=("willful", "mean"),
    ).reset_index()
    modal_soc = (g.soc2.agg(lambda s: s.mode().iat[0] if len(s.mode()) else np.nan)
                 .rename("soc2_modal").reset_index())
    return agg.merge(modal_soc, on=["key", "fy"], how="left")


agg = aggregate_lca(lca)
agg.to_csv(DATA / "lca_employer_year.csv", index=False)

describe_frame(agg, "LCA aggregated to employer-year", keys=["key", "fy"])
print()
print(agg.groupby("fy").agg(cells=("key", "size"),
                            filings=("lca_filings", "sum")).to_string())

--- LCA aggregated to employer-year ---
    rows: 330,042   columns: 13
    distinct key: 150,572
    distinct fy: 6
    columns with missing values: 5 (worst: soc2_modal at 11.6%)

      cells  filings
fy                  
2017  60154   582501
2018  59272   611156
2019  58502   624690
2020  50157   550138
2021  48812   784481
2022  53145   597093


## 4. The join

**Merge 4 of 4 — the one the whole project depends on.**

USCIS and DOL share **no employer identifier**: no EIN, no registration number. The only
common field is the employer name as typed, and the two agencies type it differently:

| USCIS | DOL |
|---|---|
| `UNIV OF N CAROLINA AT CHARLOTTE` | `UNIVERSITY OF NORTH CAROLINA AT CHARLOTTE` |
| `COGNIZANT TECH SOLNS US CORP` | `COGNIZANT TECHNOLOGY SOLUTIONS U.S. CORPORATION` |

Both sides therefore pass through the same `canon()` function (see `code/src/names.py`),
which uppercases, strips punctuation, maps long forms onto short forms, and drops
corporate suffixes. The join is a **left** join on the exact pair (key, fiscal year), so
unmatched USCIS rows are kept with a missing wage block rather than silently dropped.

In [11]:
panel = pd.read_csv(DATA / "analysis_panel.csv", low_memory=False)
panel["key"] = panel.employer.map(canon)

print("BEFORE merge 4")
describe_frame(panel, "left side: USCIS panel", keys=["key", "fy"])
describe_frame(agg, "right side: LCA employer-years", keys=["key", "fy"])
print(f"\n    keys in common: "
      f"{len(set(panel.key) & set(agg.key)):,}")
print(f"    canonicaliser example: {canon('COGNIZANT TECHNOLOGY SOLUTIONS U.S. CORPORATION')!r}")
print(f"                           {canon('COGNIZANT TECH SOLNS US CORP')!r}")

before_rows = len(panel)
merged = panel.merge(agg, on=["key", "fy"], how="left")

BEFORE merge 4
--- left side: USCIS panel ---
    rows: 184,301   columns: 25
    distinct key: 100,223
    distinct fy: 7
    columns with missing values: 2 (worst: prev_denials at 58.8%)
--- right side: LCA employer-years ---
    rows: 330,042   columns: 13
    distinct key: 150,572
    distinct fy: 6
    columns with missing values: 5 (worst: soc2_modal at 11.6%)



    keys in common: 76,779
    canonicaliser example: 'COGNIZANT TECH SOLNS U S'
                           'COGNIZANT TECH SOLNS'


In [12]:
print("AFTER merge 4")
print(f"    rows: {before_rows:,} -> {len(merged):,} "
      f"({'no duplication' if len(merged) == before_rows else 'ROW COUNT CHANGED'})")
print()
print("All rows (FY2017-FY2023):")
merge_report(merged, "lca_filings", weight_col="initial_total")

print()
print("Modelling window only (FY2017-FY2022) -- the headline figure.")
print("FY2023 matches at 0% by construction: DOL LCA coverage stops at FY2022.")
window = merged[merged.fy.isin(MODEL_YEARS)]
merge_report(window, "lca_filings", weight_col="initial_total", by="fy")

AFTER merge 4
    rows: 184,301 -> 184,301 (no duplication)

All rows (FY2017-FY2023):
    rows matched: 125,678 / 184,301  (68.2%)
    weighted by initial_total: 82.0%

Modelling window only (FY2017-FY2022) -- the headline figure.
FY2023 matches at 0% by construction: DOL LCA coverage stops at FY2022.
    rows matched: 125,678 / 172,993  (72.6%)
    weighted by initial_total: 86.0%
         rows  matched_pct  weighted_pct
fy                                      
2017  23273.0         72.9          82.1
2018  29102.0         68.0          82.3
2019  34854.0         68.4          85.9
2020  28587.0         72.4          87.3
2021  27074.0         74.3          87.4
2022  30103.0         80.7          89.4


0          True
1          True
2          True
3         False
4          True
          ...  
172988     True
172989     True
172990     True
172991     True
172992     True
Name: lca_filings, Length: 172993, dtype: bool

### Who fails to match

The failures are **not random**, and this is the most important caveat in the project.
Match rate rises monotonically with employer size, so the wage subsample tilts toward
larger sponsors. `04_models` handles this by refitting the no-wage baseline on the
matched subsample, so the with-versus-without comparison isolates the features rather
than the sample.

In [13]:
w = window.copy()
w["size_decile"] = pd.qcut(w.initial_total.rank(method="first"), 10, labels=False) + 1
by_size = w.groupby("size_decile").apply(
    lambda g: pd.Series({
        "min_petitions": g.initial_total.min(),
        "max_petitions": g.initial_total.max(),
        "matched_pct": g.lca_filings.notna().mean() * 100,
    }), include_groups=False)
print("Match rate by employer size decile:")
print(by_size.round(1).to_string())

print("\nLargest unmatched employers by petition volume:")
unmatched = window[window.lca_filings.isna()]
print(unmatched.groupby("employer").initial_total.sum().nlargest(10).to_string())

Match rate by employer size decile:
             min_petitions  max_petitions  matched_pct
size_decile                                           
1                      1.0            1.0         66.8
2                      1.0            1.0         59.7
3                      1.0            1.0         59.3
4                      1.0            1.0         64.2
5                      1.0            1.0         67.9
6                      1.0            1.0         75.7
7                      1.0            2.0         77.9
8                      2.0            3.0         80.2
9                      3.0            7.0         86.2
10                     7.0         5498.0         88.6

Largest unmatched employers by petition volume:
employer
COGNIZANT TECH SOLNS US CORP          3618
ERNST YOUNG US LLP                    1763
ERNST & YOUNG US LLP                  1304
PRICEWATERHOUSECOOPERS ADVISORY SE    1303
CITIBANK NA                           1029
BANK OF AMERICA NA             

Those are household names that certainly do file LCAs, which confirms the failures are
**join artifacts, not employers genuinely absent from the DOL data**. Two causes,
measured below: fiscal-year misalignment (an LCA is certified months before the petition
it supports, so it often falls in the prior fiscal year) and residual name variation the
fixed abbreviation dictionary does not cover.

In [14]:
# Diagnosis (a): is the employer present in the LCA data under a DIFFERENT year?
present_any_year = unmatched.key.isin(set(agg.key))
print(f"Unmatched rows whose key IS in the LCA data under another fiscal year: "
      f"{present_any_year.mean():.1%}")
print(f"    weighted by petitions: "
      f"{unmatched.initial_total[present_any_year].sum() / unmatched.initial_total.sum():.1%}")

# What a relaxed +/- 1 year join would recover, for the record.
keys = agg[["key", "fy"]].drop_duplicates()
cumulative = window.lca_filings.notna().values
for offset in (1, -1):
    shifted = keys.assign(fy=keys.fy + offset, hit=1)
    hit = window.merge(shifted, on=["key", "fy"], how="left").hit.notna().values
    cumulative = cumulative | hit
print(f"\nAllowing a +/-1 year join would reach "
      f"{cumulative.mean():.1%} of rows, "
      f"{window.initial_total[cumulative].sum() / window.initial_total.sum():.1%} of petitions.")
print("Not adopted here: it changes what the wage variables measure. Documented as future work.")

Unmatched rows whose key IS in the LCA data under another fiscal year: 41.4%


    weighted by petitions: 33.6%



Allowing a +/-1 year join would reach 83.0% of rows, 89.8% of petitions.
Not adopted here: it changes what the wage variables measure. Documented as future work.


In [15]:
merged.to_csv(DATA / "analysis_panel_wages.csv", index=False)
describe_frame(merged, "FINAL joined panel (written to disk)", keys=["employer", "key", "fy"])

wage_cols = ["wage_ratio_med", "wage_annual_med", "pw_annual_med",
             "share_low_level", "share_agent", "share_dependent"]
print("\nWage block, matched rows only:")
merged.loc[merged.lca_filings.notna(), wage_cols].describe().round(3)

--- FINAL joined panel (written to disk) ---
    rows: 184,301   columns: 36
    distinct employer: 107,466
    distinct key: 100,223
    distinct fy: 7
    columns with missing values: 13 (worst: prev_denials at 58.8%)

Wage block, matched rows only:


,wage_ratio_med,wage_annual_med,pw_annual_med,share_low_level,share_agent,share_dependent
count,125427.000,125427.000,125427.000,123417.000,125678.000,125678.000
mean,1.106,94050.083,83786.191,0.751,0.891,0.115
std,0.235,43558.693,29725.551,0.332,0.284,0.309
min,0.917,15600.000,15080.000,0.000,0.000,0.000
25%,1.000,68608.700,63575.400,0.500,1.000,0.000
50%,1.036,86070.200,80288.000,1.000,1.000,0.000
75%,1.131,107000.000,96637.000,1.000,1.000,0.000
max,12.000,1350000.000,363376.000,1.000,1.000,1.000


Two numbers to carry forward. The median `wage_ratio_med` sits just above 1.0 — the
typical employer offers barely more than the prevailing wage — and the median
`share_low_level` is 1.0, meaning that for the typical employer-year *every* certified
position sits at wage level I or II. That is the Costa and Hira (2020) wage-arbitrage
finding reproduced at employer grain.

**Next:** `03_eda.ipynb`.